# 多轮对话聊天机器人

In [13]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import (
HumanMessage, # 用户消息
SystemMessage, # 系统消息
AIMessage, # AI消息
)


def keep_recent_messages(messages_, max_pairs=3):
    """
    保留最近的 N 轮对话
    :param messages_:
    :param max_pairs: 保留的对话轮数（每轮 = user + assistant）
    :return:
    """
    # 分离 system 和对话
    system_msgs = [m for m in messages_ if m.type == 'system']
    conversation_msgs = [m for m in messages_ if m.type != 'system']

    # 只保留最近的
    recent_msgs = conversation_msgs[-(max_pairs * 2):]

    # 返回 system + 最近对话
    return system_msgs + recent_msgs


MAX_PAIRS_HISTORY = 10
EXIT_WORD = 'quit'

#  读取 .env 配置文件中的信息
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL')
DEEPSEEK_MODEL = os.getenv('DEEPSEEK_MODEL')

# 获取大模型
model = init_chat_model(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

# 消息列表
messages = [
        SystemMessage(content='你是K姐姐，K公司的数字员工，也是一名耐心、友好的智能助手。我会用自然、清晰的方式回答用户问题。')
]

# 启动提示
print(f"✨ 请输入问题，输入 {EXIT_WORD} 结束对话\n")

# 多轮对话主循环
# 轮次记录
i = 1
while True:
    print('\n','='* 10,f'-> 第{i} 轮对话开始 <-','='* 10,'\n')
    user_input = input('🥷请输入:')

    # 退出判断
    if user_input.lower() == EXIT_WORD:
        print('😁 对话已结束，欢迎下次再来！')
        break

    # 追加用户信息
    messages.append(HumanMessage(content=user_input))

    # 流式输出模型回复
    print(' 🍃K姐姐：',end='', flush=True)

    reply_content = ''

    # 优化历史记忆
    memory_messages = keep_recent_messages(messages,max_pairs=MAX_PAIRS_HISTORY)
    # 控制发送给模型的消息长度
    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content, end='', flush=True)
            reply_content += chunk.content
    print('\n', '=' * 10, f'-> 第 {i} 轮对话结束 <-', '=' * 10, '\n')
    i += 1

    # 追加 AI 回复
    messages.append(AIMessage(content=reply_content))

TypeError: unsupported operand type(s) for |: 'SystemMessage' and 'NoneType'